## FRAP analyses

In [ ]:
from pathlib import Path
import sys
import subprocess
from microlive.pipelines.pipeline_FRAP import *
from microlive.imports import *
from microlive import microscopy as mi
import subprocess
segmentation_model_path = Path().resolve().parents[1] / 'modeling' / 'cellpose_models' / 'cellpose_models' / 'FRAP_nuclei_model' / 'models' / 'cellpose_1728581750.581418'
current_dir = Path().resolve()


In [ ]:
data_folder_path = Path('/Users/nzlab-la/Desktop/20260428 FRAP Analysis') # path to your data. 
list_folders_testing = read_lif_files_in_folder(data_folder_path)
print(list_folders_testing)

In [ ]:
frap_time = 10  # Time of the bleach event in seconds (frame index where photobleaching occurs)

starting_changing_frame = 11  # Frame index where the time interval between frames starts increasing
step_size_increase = 5  # Time step multiplier applied after starting_changing_frame (seconds per frame)
list_selected_frame_values_real_time = [0, 10, 50, 100, 200]  # Real-time values (seconds) for representative frame snapshots

stable_FRAP_channel = 0  # Channel index used as a stable reference (not bleached)
FRAP_channel_to_quantify = 0  # Channel index to measure fluorescence recovery
radius_roi_size_px = 12  # Radius of the circular ROI around the bleach spot in pixels
save_individual_dataframes = False  # If True, save a separate CSV for each image
fit_model_considering_immobile_fraction = False  # If True, use immobile-fraction model for curve fitting
use_frap_time_for_roi_detection = True  # If True, use the bleach frame to auto-detect the ROI position
pretrained_model_segmentation = segmentation_model_path  # Path to custom Cellpose model (None or "auto" for defaults)
roi_drop_threshold = 0.85  # Fraction of baseline for ROI validation (0.6 = 40% drop required; use 0.85 for subtle FRAP)


In [ ]:


# Get absolute path to the CLI script to prevent "File Not Found" on Windows
script_path = Path.cwd().joinpath('FRAP_line_command.py')
list_selected_frames_str = ','.join(map(str, list_selected_frame_values_real_time))
for folder in list_folders_testing:
    cmd = [
        sys.executable, 
        str(script_path), 
        str(folder),
        str(frap_time), 
        str(stable_FRAP_channel), 
        str(FRAP_channel_to_quantify),
        str(radius_roi_size_px), 
        str(save_individual_dataframes),
        str(fit_model_considering_immobile_fraction), 
        str(starting_changing_frame),
        str(step_size_increase), 
        str(use_frap_time_for_roi_detection),
        str(pretrained_model_segmentation) if pretrained_model_segmentation is not None else 'None',
        str(roi_drop_threshold),
        list_selected_frames_str
    ]
    print(f'Processing: {folder.stem}')
    subprocess.run(cmd)

print('All folders processed.')
